In [ ]:
#Requires BioMedParse key from huggingface https://huggingface.co/microsoft/BiomedParse
from huggingface_hub import notebook_login
notebook_login()


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [2]:
"""
Inference for BioMedParse
Must recieve access from Microsoft through huggingface to download the model
Change Text Prompt in Cell 4
"""

# Install dependencies
!pip install -q torch torchvision torchaudio
!pip install -q hydra-core huggingface_hub scikit-image nibabel pandas scikit-learn scipy
!pip install -q timm transformers opencv-python Pillow

!pip install -q 'git+https://github.com/facebookresearch/detectron2.git'

!pip install -q einops fvcore iopath



# Mount Drive
from google.colab import drive
drive.mount('/content/drive')

# Download dataset from Kaggle

import os

# Install kaggle
!pip install -q kaggle

# Create kaggle directory
!mkdir -p ~/.kaggle

# Check if kaggle.json already exists
if not os.path.exists('/root/.kaggle/kaggle.json'):
    print("\n Upload kaggle.json file:")
    from google.colab import files
    uploaded = files.upload()

    # Move kaggle.json to correct location
    !cp kaggle.json ~/.kaggle/
    !chmod 600 ~/.kaggle/kaggle.json

# Download dataset
DATASET_PATH = "/content/dataset/lgg-mri-segmentation/kaggle_3m"
if not os.path.exists(DATASET_PATH):
    !kaggle datasets download -d mateuszbuda/lgg-mri-segmentation
    !unzip -q lgg-mri-segmentation.zip -d /content/dataset

    # Verify extraction
    if os.path.exists(DATASET_PATH):
        print("Dataset downloaded ")
    else:
        # Check alternate locations
        import glob
        data_csv_files = glob.glob("/content/dataset/**/data.csv", recursive=True)
        if data_csv_files:
            DATASET_PATH = os.path.dirname(data_csv_files[0])
            print(f"Dataset found at: {DATASET_PATH}")
else:
    print("Dataset already exists")

print(f"\nDataset location: {DATASET_PATH}")

# Clone BiomedParse repo
import os
if not os.path.exists('/content/BiomedParse'):
    !git clone https://github.com/microsoft/BiomedParse.git

# Add to path
import sys
sys.path.insert(0, '/content/BiomedParse/src')
print("BiomedParse added to path")




   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.5/154.5 kB 8.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 4.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.9/91.9 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 78.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.2/55.2 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 269.8/269.8 kB 42.0 MB/s eta 0:00:00


ValueError: mount failed

In [ ]:
#Load Model

import torch
import numpy as np
import nibabel as nib
import pandas as pd
from PIL import Image
from glob import glob
import torch.nn.functional as F
import hydra
from hydra import compose
from hydra.core.global_hydra import GlobalHydra
from huggingface_hub import hf_hub_download
from sklearn.metrics import f1_score, jaccard_score

import os

os.chdir('/content/BiomedParse')
print(f"  Working directory: {os.getcwd()}")

# Verify configs exist
config_path = os.path.abspath('configs/model')
if os.path.exists(config_path):
    print(f"  Configs at: {config_path}")
else:
    print(f"  Configs directory not found at {config_path}!")
    raise FileNotFoundError(f"configs/model directory not found at {config_path}")

# Import BiomedParse utilities
import sys
if '/content/BiomedParse/src' not in sys.path:
    sys.path.insert(0, '/content/BiomedParse/src')

from utils import process_input, process_output
from inference import postprocess, merge_multiclass_masks

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"  Device: {device}")

# Initialize Hydra with ABSOLUTE path
GlobalHydra.instance().clear()
hydra.initialize_config_dir(
    config_dir=config_path,
    job_name="batch_prediction",
    version_base=None
)
cfg = compose(config_name="biomedparse_3D")
model = hydra.utils.instantiate(cfg, _convert_="object")

# Download and load checkpoint
checkpoint_path = hf_hub_download(
    repo_id="microsoft/BiomedParse",
    filename="biomedparse_v2.ckpt"
)
model.load_pretrained(checkpoint_path)
model = model.to(device).eval()


In [ ]:
#Configuration

# Paths
OUTPUT_PATH = "/content/drive/MyDrive/SAM_results_biomedparse"
os.makedirs(OUTPUT_PATH, exist_ok=True)

# Text prompt for brain tumor
TEXT_PROMPT = "Enhancing tumor core"

print(f"  Dataset: {DATASET_PATH}")
print(f"  Output: {OUTPUT_PATH}")
print(f"  Prompt: {TEXT_PROMPT}")

  Dataset: /content/dataset/lgg-mri-segmentation/kaggle_3m
  Output: /content/drive/MyDrive/SAM_results_biomedparse
  Prompt: Enhancing tumor core


In [ ]:
#Functions

def segment_3d_volume(volume, text_prompt, slice_batch_size=4):
    """
    Segment a 3D volume using BiomedParse with a hard text prompt.

    Args:
        volume: numpy array of shape (D, H, W) - depth, height, width
        text_prompt: text description of what to segment
        slice_batch_size: number of slices to process at once

    Returns:
        segmentation mask of same shape as input
    """
    # Prepare input
    imgs, pad_width, padded_size, valid_axis = process_input(volume, 512)
    imgs = imgs.to(device).int()

    input_tensor = {
        "image": imgs.unsqueeze(0),
        "text": [text_prompt],
    }

    # Run inference
    with torch.no_grad():
        outputs = model(input_tensor)

    mask_preds = process_output(outputs, pad_width, padded_size, valid_axis, volume.shape)

    return mask_preds


def load_patient_volume_from_tif(patient_dir):
    """
    Load all TIF slices into a 3D volume

    Returns:
        volume: numpy array (D, H, W)
        image_files: list of image file paths
    """
    image_files = sorted([
        f for f in glob(os.path.join(patient_dir, "*.tif"))
        if f[-5].isdigit() and "_mask" not in f
    ])

    if len(image_files) == 0:
        return None, None

    slices = []
    for img_file in image_files:
        img = np.array(Image.open(img_file).convert('L'))
        slices.append(img)

    volume = np.stack(slices, axis=0)

    return volume, image_files


def process_patient(patient_dir, patient_id, text_prompt):
    """
    Process a single patient
    """
    print(f"  Loading volume...")
    volume, image_files = load_patient_volume_from_tif(patient_dir)

    if volume is None:
        print(f"    No images found")
        return None

    print(f"    Volume shape: {volume.shape}")

    print(f"  Running BiomedParse segmentation...")
    pred_mask = segment_3d_volume(volume, text_prompt, slice_batch_size=4)

    print(f"    Segmentation shape: {pred_mask.shape}")

    return pred_mask, image_files


In [ ]:
#Process Patients

# Load patient info
data_csv = os.path.join(DATASET_PATH, "data.csv")
patient_info = pd.read_csv(data_csv)
patient_dirs = sorted(glob(os.path.join(DATASET_PATH, "TCGA_*")))

print(f"Patients in dataset: {len(patient_info)}")
print(f"Patient directories found: {len(patient_dirs)}")
print(TEXT_PROMPT)

results = []

for idx, patient_dir in enumerate(patient_dirs, 1):
    full_patient_id = os.path.basename(patient_dir)
    short_patient_id = '_'.join(full_patient_id.split('_')[:3])

    print(f"\n[{idx}/{len(patient_dirs)}] {short_patient_id}")

    # Get demographics
    patient_row = patient_info[patient_info['Patient'] == short_patient_id]
    if len(patient_row) == 0:
        print("  No demographics, skipping")
        continue

    race = patient_row.iloc[0]['race']
    gender = patient_row.iloc[0]['gender']
    print(f"  Race: {race}, Gender: {gender}")

    try:
        # Process patient
        result = process_patient(patient_dir, short_patient_id, TEXT_PROMPT)

        if result is None:
            continue

        pred_volume, image_files = result

        print(f"  Segmentation complete")

        # Save NIfTI
        affine = np.eye(4)
        out_path = os.path.join(OUTPUT_PATH, f"{short_patient_id}_biomedparse_predictions.nii.gz")
        nib.save(nib.Nifti1Image(pred_volume, affine), out_path)
        print(f"  Saved: {out_path}")

        # Calculate metrics against ground truth
        gt_pairs = []
        for img_file in image_files:
            base = os.path.splitext(img_file)[0]
            mask_file = f"{base}_mask.tif"
            if os.path.exists(mask_file):
                gt_pairs.append((img_file, mask_file))

        ious = []
        f1s = []

        for z, (_, mask_file) in enumerate(gt_pairs):
            if z >= pred_volume.shape[0]:
                break

            # Load GT
            gt_slice = np.array(Image.open(mask_file).convert("L"))
            gt_binary = (gt_slice > 0).astype(np.uint8)

            # Get prediction slice
            pred_slice = pred_volume[z]

            # Resize if needed
            if gt_binary.shape != pred_slice.shape:
                from scipy.ndimage import zoom
                scale_factors = [gt_binary.shape[0] / pred_slice.shape[0],
                               gt_binary.shape[1] / pred_slice.shape[1]]
                pred_slice = zoom(pred_slice, scale_factors, order=0).astype(np.uint8)

            # Calculate metrics (only if there's something to measure)
            gt_sum = gt_binary.sum()
            pred_sum = pred_slice.sum()

            # Case 1: both empty → perfect score
            if gt_sum == 0 and pred_sum == 0:
                ious.append(1.0)
                f1s.append(1.0)

            # Case 2: one empty, one not → zero score
            elif gt_sum == 0 and pred_sum > 0:
                ious.append(0.0)
                f1s.append(0.0)

            elif gt_sum > 0 and pred_sum == 0:
                ious.append(0.0)
                f1s.append(0.0)

            # Case 3: both non-empty → compute normally
            else:
                ious.append(
                    jaccard_score(
                        gt_binary.flatten(),
                        pred_slice.flatten(),
                        zero_division=0
                    )
                )
                f1s.append(
                    f1_score(
                        gt_binary.flatten(),
                        pred_slice.flatten(),
                        zero_division=0
                    )
                )


        mean_iou = np.mean(ious) if ious else 0
        mean_f1 = np.mean(f1s) if f1s else 0

        print(f"  IoU: {mean_iou:.4f}, F1: {mean_f1:.4f}")

        results.append({
            'Patient': short_patient_id,
            'Race': race,
            'Gender': gender,
            'IoU': mean_iou,
            'F1': mean_f1,
            'Num_Slices': len(ious),
            'Prediction_Path': out_path
        })

    except Exception as e:
        print(f"   Error: {str(e)[:200]}")
        import traceback
        traceback.print_exc()
        continue

Patients in dataset: 110
Patient directories found: 110
Enhancing tumor core

[1/110] TCGA_CS_4941
  Race: 3.0, Gender: 2.0
  Loading volume...
    Volume shape: (23, 256, 256)
  Running BiomedParse segmentation...
    Segmentation shape: (23, 256, 256)
  Segmentation complete
  Saved: /content/drive/MyDrive/SAM_results_biomedparse/TCGA_CS_4941_biomedparse_predictions.nii.gz
  IoU: 0.6522, F1: 0.6522

[2/110] TCGA_CS_4942
  Race: 2.0, Gender: 1.0
  Loading volume...
    Volume shape: (20, 256, 256)
  Running BiomedParse segmentation...
    Segmentation shape: (20, 256, 256)
  Segmentation complete
  Saved: /content/drive/MyDrive/SAM_results_biomedparse/TCGA_CS_4942_biomedparse_predictions.nii.gz
  IoU: 0.7000, F1: 0.7000

[3/110] TCGA_CS_4943
  Race: 3.0, Gender: 2.0
  Loading volume...
    Volume shape: (20, 256, 256)
  Running BiomedParse segmentation...
    Segmentation shape: (20, 256, 256)
  Segmentation complete
  Saved: /content/drive/MyDrive/SAM_results_biomedparse/TCGA_CS_4943

In [ ]:
#Save Results

if len(results) > 0:
    results_df = pd.DataFrame(results)

    overall_iou = results_df["IoU"].mean() if len(results_df) > 0 else 0.0
    overall_f1  = results_df["F1"].mean()  if len(results_df) > 0 else 0.0

    print(f"Overall IoU: {overall_iou:.4f}")
    print(f"Overall F1 : {overall_f1:.4f}")
    print(f"Patients   : {len(results_df)}")

    csv_path = os.path.join(OUTPUT_PATH, "biomedparse_patient_metrics.csv")
    results_df.to_csv(csv_path, index=False)
    print("\nSaved:", csv_path)

    race_stats = (
        results_df
        .groupby("Race")[["IoU","F1"]]
        .agg(["mean","std","count"])
        .round(4)
    )

    print("\n" + "="*70)
    print("RACE LEVEL METRICS")
    print("="*70)
    print(race_stats)

    race_stats.to_csv(
        os.path.join(OUTPUT_PATH, "biomedparse_race_metrics.csv")
    )

    gender_stats = (
        results_df
        .groupby("Gender")[["IoU","F1"]]
        .agg(["mean","std","count"])
        .round(4)
    )

    print("\n" + "="*70)
    print("GENDER LEVEL METRICS")
    print("="*70)
    print(gender_stats)

    gender_stats.to_csv(
        os.path.join(OUTPUT_PATH, "biomedparse_gender_metrics.csv")
    )

    summary_path = os.path.join(OUTPUT_PATH, "biomedparse_summary.txt")

    with open(summary_path, "w") as f:
        f.write("OVERALL METRICS\n")
        f.write("-"*70 + "\n")
        f.write(f"Mean IoU: {overall_iou:.4f}\n")
        f.write(f"Mean F1 : {overall_f1:.4f}\n")
        f.write(f"Patients: {len(results_df)}\n\n")

        f.write("RACE-LEVEL METRICS\n")
        f.write("-"*70 + "\n")
        f.write(race_stats.to_string())
        f.write("\n\n")

        f.write("GENDER-LEVEL METRICS\n")
        f.write("-"*70 + "\n")
        f.write(gender_stats.to_string())
        f.write("\n")

    print("Saved summary:", summary_path)
    print("Results directory:", OUTPUT_PATH)

else:
    print("No results to save")


Saved: /content/drive/MyDrive/SAM_results_biomedparse/biomedparse_patient_metrics.csv

OVERALL METRICS
  Mean IoU: 0.6441
  Mean F1:  0.6447
  Patients: 110

RACE-LEVEL METRICS
         IoU                    F1              
        mean     std count    mean     std count
Race                                            
2.0   0.6475  0.1100    10  0.6475  0.1100    10
3.0   0.6419  0.1137    98  0.6426  0.1138    98

GENDER-LEVEL METRICS
           IoU                    F1              
          mean     std count    mean     std count
Gender                                            
1.0     0.6542  0.1053    56  0.6545  0.1055    56
2.0     0.6320  0.1201    53  0.6329  0.1202    53
 Saved: /content/drive/MyDrive/SAM_results_biomedparse/biomedparse_summary.txt

🎉 BATCH PROCESSING COMPLETE!

Results saved to: /content/drive/MyDrive/SAM_results_biomedparse

Files created:
  - biomedparse_patient_metrics.csv
  - biomedparse_race_metrics.csv
  - biomedparse_gender_metrics.csv
  - bi